<a href="https://colab.research.google.com/github/ivanduzunov/AI-Agents-and-Workflows-for-Developers/blob/main/Multi_Agent_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install q langchain langchain-openai

In [35]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Send
from typing import TypedDict, Literal, List, Annotated, Dict
from IPython.display import Image

In [37]:
def merge_dicts(a: Dict, b: Dict) -> Dict:
  return {**a, **b}

def merge_lists(a: list[str], b: list[str]) -> List:
  return [*a, *b]

class EditorReviewState(TypedDict):
    verdict: Literal["approve", "reject"]
    comments: List[str]

class MarketingState(TypedDict):
    product_brief: str
    target_platform: str
    research: str
    draft: str
    review: EditorReviewState
    revision_cycles: int

class SystemState(TypedDict):
    promt: str
    outcome: Annotated[Dict[str, str], merge_dicts]

In [26]:
def empty_fn(state: MarketingState):
  pass

def analyst(state: MarketingState):
  return {"research": "The analyst works..."}

def writer(state: MarketingState):
  # todo: AI
  return {"draft": "The writer works..."}

def editor(state: MarketingState):
  revision_cycles = state.get("revision_cycles", 0)
  if revision_cycles >= 3:
    # todo: add interrupts
    return {"review": {"verdict": "approve", "comments": ["Forced state"]}}
  return {"review": {"verdict": "reject", "comments": []}, "revision_cycles": revision_cycles + 1}

def editor_path(state: MarketingState):
  review = state.get("review", {})
  verdict = review.get("verdict")
  if verdict == "approve":
    return END
  else:
    return "Writer"

In [5]:
marketing_graph_builder = StateGraph(MarketingState)
marketing_graph_builder.add_node("Analyst", analyst)
marketing_graph_builder.add_node("Writer", writer)
marketing_graph_builder.add_node("Editor", editor)

marketing_graph_builder.add_edge(START, "Analyst")
marketing_graph_builder.add_edge("Analyst", "Writer")
marketing_graph_builder.add_edge("Writer", "Editor")
marketing_graph_builder.add_conditional_edges("Editor", editor_path, ["Writer", END])

marketing_graph = marketing_graph_builder.compile(debug=True)

In [ ]:
display(Image(marketing_graph.get_graph(xray=True).draw_mermaid_png(), format="png"))

In [38]:
def supervisor(state: SystemState):
  pass

def produce_outcome(state: MarketingState):
  target_platform = state.get("target_platform")
  final_post = state.get("draft")
  return {"outcome": {target_platform: final_post}}

def delegate_tasks(state: SystemState):
  return [Send("Marketing Team", {"product_brief": "We are launching a new eco-friendly smart water bottle. Target: gym-goers.", "target_platform": "LinkedIn"}),
          Send("Marketing Team", {"product_brief": "We are launching a new eco-friendly smart water bottle. Target: gym-goers.", "target_platform": "Facebook"})]

In [39]:
system_graph_buildr = StateGraph(SystemState)
system_graph_buildr.add_node("Supervisor", supervisor)
system_graph_buildr.add_node("Marketing Team", marketing_graph | produce_outcome)

system_graph_buildr.add_edge(START, "Supervisor")
system_graph_buildr.add_conditional_edges("Supervisor", delegate_tasks, ["Marketing Team", END])
system_graph_buildr.add_edge("Marketing Team", END)

system_graph = system_graph_buildr.compile(debug=True)

In [ ]:
display(Image(system_graph.get_graph(xray=True).draw_mermaid_png(), format="png"))

In [ ]:
marketing_graph.invoke(input={"product_brief": "We are launching a new eco-friendly smart water bottle. Target: gym-goers.", "target_platform": "LinkedIn"})

In [ ]:
system_graph.invoke(input={"prompt": "We are launching a new eco-friendly smart water bottle. Target platform: LinkedIn."})

In [ ]:
# EXAMPLE PART

In [ ]:
class SystemState(TypedDict):
    pass

def main(state: SystemState):
  print("Executing node MAIN")

def a(state: SystemState):
  print("Executing node A")

def b1(state: SystemState):
  print("Executing node B1")

def b2(state: SystemState):
  print("Executing node B2")

def finish(state: SystemState):
  print("Executing node FINISH")

In [ ]:
checkpointer = InMemorySaver()

In [ ]:
sub_graph_builder = StateGraph(SystemState)
sub_graph_builder.add_node("B1", b1)
sub_graph_builder.add_node("B2", b2)

sub_graph_builder.add_edge(START, "B1")
sub_graph_builder.add_edge("B1", "B2")
sub_graph_builder.add_edge("B2", END)

sub_graph = sub_graph_builder.compile()

In [ ]:
graph_builder = StateGraph(SystemState)
graph_builder.add_node("main", main)
graph_builder.add_node("A", a)
graph_builder.add_node("B", sub_graph)
graph_builder.add_node("finish", finish)


graph_builder.add_edge(START, "main")
graph_builder.add_edge("main", "A")
graph_builder.add_edge("main", "B")
graph_builder.add_edge("A", "finish")
graph_builder.add_edge("B", "finish")


graph = graph_builder.compile(checkpointer=checkpointer)

In [ ]:
display(Image(graph.get_graph(xray=True).draw_mermaid_png(), format="png"))

In [ ]:
thread1_config = {"configurable": {"thread_id": "tr_1"}}
graph.invoke(input={}, config=thread1_config)

In [ ]:
list(checkpointer.list(thread1_config))

In [ ]:
list(graph.get_state_history(thread1_config))